In [ ]:
# Setup
# %pip install -q openai python-dotenv


## 1) Import


In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json


## 2) Load environment variables - please read instructions carefully


In [ ]:
# if you are running in local, uncomment below line. also make sure you shall have a .env file
# load_dotenv()

True

In [ ]:
# if you are running in google colab, uncomment below line. and replace "Your_API_Key" with your own openAI API key
#os.environ["OPENAI_API_KEY"] = "Your_API_Key"

In [2]:
# llm_name = "gpt-4"
# openai_key = os.getenv("OPENAI_API_KEY")
# client = OpenAI(api_key=openai_key)

client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # placeholder, Ollama ignores this
)

llm_name = "qwen3"


## 3) Tool


In [3]:
# Implement the functions actions
def estimate_trip_cost(
    days: int,
    travelers: int,
    comfort: str = "mid",
):
    """
    Estimate a rough trip budget (SGD) using simple heuristics.
    comfort: budget | mid | premium
    Returns a breakdown and total estimate in SGD.
    """
    if days <= 0 or travelers <= 0:
        raise ValueError("days and travelers must be > 0")

    comfort = comfort.lower().strip()
    if comfort not in {"budget", "mid", "premium"}:
        raise ValueError("comfort must be one of: budget, mid, premium")

    # Very rough per-person-per-day estimates (SGD) excluding flights
    lodging_per_person_day = {"budget": 60, "mid": 140, "premium": 300}[comfort]
    food_per_person_day = {"budget": 30, "mid": 60, "premium": 120}[comfort]
    local_transport_per_person_day = {"budget": 10, "mid": 20, "premium": 50}[comfort]
    activities_per_person_day = {"budget": 20, "mid": 50, "premium": 120}[comfort]

    lodging = lodging_per_person_day * travelers * days
    food = food_per_person_day * travelers * days
    transport = local_transport_per_person_day * travelers * days
    activities = activities_per_person_day * travelers * days

    subtotal = lodging + food + transport + activities
    contingency = round(subtotal * 0.12)  # 12% buffer
    total = subtotal + contingency

    return f"total_estimate: {total}"

known_actions = {"estimate_trip_cost": estimate_trip_cost}


## 4) Tool Schema


In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "estimate_trip_cost",
            "description": "Estimate a rough trip budget (SGD) using simple heuristics. comfort: budget | mid | premium. Returns a breakdown and total estimate in SGD.",
            "parameters": {
                "type": "object",
                "properties": {
                    "days": {
                        "type": "integer",
                        "description": "Number of travel days"
                    },
                    "travelers": {
                        "type": "integer",
                        "description": "Number of travelers"
                    },
                    "comfort": {
                        "type": "string",
                        "description": "Travel comfort level",
                        "enum": ["budget", "mid", "premium"],
                        "default": "mid"
                    }
                },
                "required": ["days", "travelers"]
            }
        }
    }
]

In [5]:
system_prompt = """
You run in a loop of Thought, Action, Observation.
At the end of the loop you output an Answer.
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you.
Observation will be the result of running those actions.

Important notes:
Do not invent any facts, numbers. Use available actions instead.

""".strip()


## 5) Agent loop - step by step


In [6]:
messages = []
user_message = "a 2-day Tokyo trip for 2 adults. Mid comfort. How much roughly will it cost?"
messages.append({"role": "system", "content": system_prompt})
messages.append({"role": "user", "content": user_message})

In [7]:
response = client.chat.completions.create(
    model=llm_name,
    temperature=0.4,
    messages=messages,
    tools=tools
)
assistant_message = response.choices[0].message
messages.append(assistant_message)

In [8]:
print("\n response from LLM ")
print("\n", response)


 response from LLM 

 ChatCompletion(id='chatcmpl-20', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_uk3dj1as', function=Function(arguments='{"comfort":"mid","days":2,"travelers":2}', name='estimate_trip_cost'), type='function', index=0)], reasoning='Okay, let\'s see. The user is asking about the cost for a 2-day Tokyo trip for two adults with mid comfort. I need to use the estimate_trip_cost function. The parameters required are comfort, days, and travelers. The comfort is mid, days are 2, and travelers are 2. I\'ll plug those into the function. Wait, does the function require any other parameters? No, the tool definition lists those three. So the arguments should be comfort: "mid", days: 2, travelers: 2. Let me make sure the JSON structure is correct. The function name is estimate_

In [9]:
tool_result = estimate_trip_cost(2, 2, "mid")
print(tool_result)

total_estimate: 1210


In [16]:
messages.append({
    "role": "tool",
    #change the tool_call_id to correct tool call ID returned in the previous AI message
    "tool_call_id": "call_uk3dj1as",
    "content": json.dumps(tool_result)
})

In [11]:
response = client.chat.completions.create(
    model=llm_name,
    temperature=0.4,
    messages=messages,
    tools=tools
)

In [12]:
print("\n response from LLM ")
print("\n", response)


 response from LLM 

 ChatCompletion(id='chatcmpl-296', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The estimated cost for a 2-day Tokyo trip for 2 adults with mid comfort is **~SGD 1210**. This is a rough approximation based on heuristic calculations for mid-range travel expenses.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning="Okay, the user asked for a rough estimate of a 2-day Tokyo trip for two adults with mid comfort. I called the estimate_trip_cost function with the parameters comfort: mid, days: 2, travelers: 2. The response came back with a total estimate of 1210 SGD. Now I need to present this answer clearly. Let me check if the function provides a breakdown, but the tool response only mentions the total. Since the user asked for a rough cost, stating the total and mentioning that it's a mid-comfort estimate should be sufficient. Maybe add a note that this

In [13]:
print(response.choices[0].message.content)

The estimated cost for a 2-day Tokyo trip for 2 adults with mid comfort is **~SGD 1210**. This is a rough approximation based on heuristic calculations for mid-range travel expenses.



## 6) Custom Agent


In [14]:
# Create an agent using OpenAI native tool calling
class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({
            "role": "user",
            "content": message
        })
        count =0;
        while True:
            response = client.chat.completions.create(
                model=llm_name,
                temperature=0.4,
                messages=self.messages,
                tools=tools
            )
            print("\n turn ", count)
            print("\n messages sent to LLM ")
            print("\n", self.messages)
            print("\n response from LLM ")
            print("\n", response)

            assistant_message = response.choices[0].message
            self.messages.append(assistant_message)

            finish_reason = response.choices[0].finish_reason
            print("\n stop_reason", finish_reason)

            if finish_reason=="stop":
                return assistant_message.content

            for tool_call in assistant_message.tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)

                tool_result = known_actions[tool_name](**tool_args)

                self.messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result)
                })
            count=count+1


In [15]:
agent = Agent(system_prompt)
question = "Which ones cost lesser, out of these 2 choices: '2-day Tokyo trip for 2 adults mid comfort' and '3-day malaysia trip for 2 adults premium comfort'?"
response = agent(question)
print(response)


 turn  0

 messages sent to LLM 

 [{'role': 'system', 'content': 'You run in a loop of Thought, Action, Observation.\nAt the end of the loop you output an Answer.\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you.\nObservation will be the result of running those actions.\n\nImportant notes:\nDo not invent any facts, numbers. Use available actions instead.'}, {'role': 'user', 'content': "Which ones cost lesser, out of these 2 choices: '2-day Tokyo trip for 2 adults mid comfort' and '3-day malaysia trip for 2 adults premium comfort'?"}]

 response from LLM 

 ChatCompletion(id='chatcmpl-546', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content='', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_6zcutdwg', function=Function(arguments='{"comfort":"mid","days":2